## Performing Inner and Left Join on Spark Data Frames

In [1]:
%run './01 Setup Data Sets to perform joins.ipynb'

24/03/27 21:37:59 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/27 21:37:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/27 21:38:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---------+-----------------------------+-------------------+---------+-------------------+
|course_id|course_title                 |course_published_dt|is_active|last_updated_ts    |
+---------+-----------------------------+-------------------+---------+-------------------+
|1        |Mastering Python             |2023-01-14         |true     |2024-01-21 20:05:46|
|2        |Data Science with R          |2023-03-01         |false    |2023-12-15 11:32:10|
|3        |Machine Learning Fundamentals|2023-05-12         |true     |2024-02-10 14:17:23|
|4        |Deep Learning with TensorFlow|2023-07-07         |true     |2024-03-14 09:25:18|
|5        |Natural Language Processing  |2023-09-02         |false    |2024-01-09 17:42:31|
+---------+-----------------------------+-------------------+---------+-------------------+

+-------+---------------+--------------+--------------------+
|user_id|user_first_name|user_last_name|          user_email|
+-------+---------------+--------------+-------

In [2]:
users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), 'user_id'). \
    select('*'). \
    show()

+-------+---------------+--------------+--------------------+--------------------+---------+----------+
|user_id|user_first_name|user_last_name|          user_email|course_enrollment_id|course_id|price_paid|
+-------+---------------+--------------+--------------------+--------------------+---------+----------+
|      1|           John|           Doe|john.doe@example.com|                   2|        2|     10.99|
|      2|           Jane|         Smith|jane.smith@exampl...|                   1|        1|      9.99|
|      2|           Jane|         Smith|jane.smith@exampl...|                   3|        3|      9.99|
|      3|          Alice|       Johnson|alice.johnson@exa...|                   4|        4|     10.99|
|      4|        Michael|      Williams|michael.williams@...|                   5|        1|      9.99|
+-------+---------------+--------------+--------------------+--------------------+---------+----------+



In [3]:
# alias does not work using this style of syntax, so alias has been removed

users_df. \
    join(course_enrollments_df, users_df.user_id == course_enrollments_df.user_id). \
    select('*'). \
    show()

+-------+---------------+--------------+--------------------+--------------------+-------+---------+----------+
|user_id|user_first_name|user_last_name|          user_email|course_enrollment_id|user_id|course_id|price_paid|
+-------+---------------+--------------+--------------------+--------------------+-------+---------+----------+
|      1|           John|           Doe|john.doe@example.com|                   2|      1|        2|     10.99|
|      2|           Jane|         Smith|jane.smith@exampl...|                   1|      2|        1|      9.99|
|      2|           Jane|         Smith|jane.smith@exampl...|                   3|      2|        3|      9.99|
|      3|          Alice|       Johnson|alice.johnson@exa...|                   4|      3|        4|     10.99|
|      4|        Michael|      Williams|michael.williams@...|                   5|      4|        1|      9.99|
+-------+---------------+--------------+--------------------+--------------------+-------+---------+----

In [4]:
# alias does not work using this style of syntax, so alias has been removed

users_df.alias('u'). \
    join(course_enrollments_df, users_df.user_id == course_enrollments_df.user_id). \
    select('u.*'). \
    show()

+-------+---------------+--------------+--------------------+
|user_id|user_first_name|user_last_name|          user_email|
+-------+---------------+--------------+--------------------+
|      1|           John|           Doe|john.doe@example.com|
|      2|           Jane|         Smith|jane.smith@exampl...|
|      2|           Jane|         Smith|jane.smith@exampl...|
|      3|          Alice|       Johnson|alice.johnson@exa...|
|      4|        Michael|      Williams|michael.williams@...|
+-------+---------------+--------------+--------------------+



In [5]:
# show number of courses enrolled by each user
# using select will not work here to display the output; rather output will be displayed on the basis of group by

users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), users_df.user_id == course_enrollments_df.user_id). \
    groupBy('u.user_id', 'u.user_first_name'). \
    count(). \
    show(truncate=False)    

# select('u.user_id', 'u.user_first_name', 'ce.course_id'). \
    

+-------+---------------+-----+
|user_id|user_first_name|count|
+-------+---------------+-----+
|1      |John           |1    |
|2      |Jane           |2    |
|3      |Alice          |1    |
|4      |Michael        |1    |
+-------+---------------+-----+



In [6]:
# Get the number of courses enrolled by each user
# The count should be zero for users who are not enrolled in any course

users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), 'user_id', 'left'). \
    groupBy('u.user_id', 'u.user_first_name', 'u.user_email'). \
    count(). \
    show(truncate=False)


+-------+---------------+----------------------------+-----+
|user_id|user_first_name|user_email                  |count|
+-------+---------------+----------------------------+-----+
|2      |Jane           |jane.smith@example.com      |2    |
|1      |John           |john.doe@example.com        |1    |
|3      |Alice          |alice.johnson@example.com   |1    |
|4      |Michael        |michael.williams@example.com|1    |
|5      |Sarah          |sarah.davis@example.com     |1    |
|6      |Ghost          |NULL                        |1    |
+-------+---------------+----------------------------+-----+



In [7]:
from pyspark.sql.functions import col, coalesce

In [8]:
# Get the number of courses enrolled by each user
# The count should be zero for users who are not enrolled in any course
# Count cannot take any argument
# Grouped object has no attribute withColumn()


users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), 'user_id', 'left'). \
    filter(col('ce.course_enrollment_id').isNotNull()). \
    groupBy('u.user_id', 'u.user_first_name', 'u.user_email'). \
    count(). \
    show(truncate=False)


+-------+---------------+----------------------------+-----+
|user_id|user_first_name|user_email                  |count|
+-------+---------------+----------------------------+-----+
|1      |John           |john.doe@example.com        |1    |
|2      |Jane           |jane.smith@example.com      |2    |
|3      |Alice          |alice.johnson@example.com   |1    |
|4      |Michael        |michael.williams@example.com|1    |
+-------+---------------+----------------------------+-----+



In [9]:
from pyspark.sql.functions import count, lit, when, sum

In [10]:
# Get the number of courses enrolled by each user
# The count should be zero for users who are not enrolled in any course
# Count cannot take any argument
# Grouped object has no attribute withColumn()
# Grouped object has no attribute withColumnRenamed()
# coalesce(col('course_enrollment_id'), lit(0))


users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), 'user_id', 'left'). \
    groupBy('u.user_id'). \
    agg(sum(when(course_enrollments_df['course_enrollment_id'].isNotNull(),1).otherwise(0)).alias('course_count')). \
    show(truncate=False)


+-------+------------+
|user_id|course_count|
+-------+------------+
|2      |2           |
|1      |1           |
|3      |1           |
|4      |1           |
|5      |0           |
|6      |0           |
+-------+------------+



In [11]:
from pyspark.sql.functions import sum, when

%time 

users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), 'user_id', 'left'). \
    groupBy('u.user_id'). \
    agg(sum(coalesce(col('course_enrollment_id'), lit(0))).alias('course_count')). \
    show(truncate=False)


CPU times: user 1e+03 ns, sys: 0 ns, total: 1e+03 ns
Wall time: 2.15 µs
+-------+------------+
|user_id|course_count|
+-------+------------+
|2      |4           |
|1      |2           |
|3      |4           |
|4      |5           |
|5      |0           |
|6      |0           |
+-------+------------+



In [13]:
from pyspark.sql.functions import sum, when

%time 

users_df.alias('u'). \
    join(course_enrollments_df.alias('ce'), 'user_id', 'left'). \
    groupBy('u.user_id'). \
    agg(sum(coalesce(col('course_enrollment_id'), lit(0))).alias('course_count')). \
    explain(False)


CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 3.58 µs
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[user_id#31L], functions=[sum(coalesce(course_enrollment_id#56L, 0))])
   +- HashAggregate(keys=[user_id#31L], functions=[partial_sum(coalesce(course_enrollment_id#56L, 0))])
      +- Project [user_id#31L, course_enrollment_id#56L]
         +- SortMergeJoin [user_id#31L], [user_id#57L], LeftOuter
            :- Sort [user_id#31L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(user_id#31L, 200), ENSURE_REQUIREMENTS, [plan_id=1298]
            :     +- Project [user_id#31L]
            :        +- Scan ExistingRDD[user_id#31L,user_first_name#32,user_last_name#33,user_email#34]
            +- Sort [user_id#57L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(user_id#57L, 200), ENSURE_REQUIREMENTS, [plan_id=1299]
                  +- Project [course_enrollment_id#56L, user_id#57L]
                     +- F